# BoC — Rationale-alignment evaluation (LLM-as-a-judge, on the side)

This notebook is a **side-channel** evaluation: it does not touch the resolution
loop. It is **trace-driven** — the Langfuse **trace** is the canonical record of
what each forecaster said. For every trace it reads the structured forecast the
predictor stamped on at run time (its `rationale`, cited signals, and predicted
distribution), compares that rationale to the Bank of Canada's **own** published
press release for that meeting, and **pushes** a structured *alignment* verdict
back to the trace as Langfuse scores — complementing the accuracy score (RPS)
with a *process* metric: was the forecaster right **for the right reasons**?

So evaluation **reads from and writes to Langfuse**, not a local prediction cache.

**Prerequisites**
1. Langfuse configured (`LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` in `.env`).
2. Press releases cached: `uv run python scripts/fetch_boc_press_releases.py`
   (covers every scheduled date back to 2009).
3. The generation cell in section 2 runs the reasoning predictors live, so a
   fresh trace exists for every meeting it scores — no prior traced run needed.

**Cutoff posture.** This notebook runs on the **protected post-2025 eval window**
(Jan 2025 – Jun 2026), the same honest origins as notebook 02 §10. They sit
at/after the model's ~January 2025 training cutoff, so the rationale being judged
reflects genuine reasoning rather than a recalled outcome — the alignment verdict
is as clean as the accuracy score there. (Pointing this at a pre-2025 backtest
would inherit the same memorisation caveat as the accuracy backtest.)

---
## 1. Setup

In [1]:
from __future__ import annotations

import warnings
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import yaml
from dotenv import load_dotenv
from IPython.display import Markdown, display  # noqa: A004


warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parents[1]
load_dotenv(ROOT / ".env")
# ── build-cli token refresh ───────────────────────────────────────────────────
import os as _os, shutil as _shutil, subprocess as _subprocess
_BUILDCLI = next(
    (str(p) for p in [Path.home() / ".local" / "bin" / "build-cli",
                       Path(_shutil.which("build-cli") or "")] if p.exists()), None,
)
if "build-cli.roche.com" in _os.environ.get("PROXY_BASE_URL", ""):
    if _BUILDCLI:
        _result = _subprocess.run([_BUILDCLI, "auth", "token"], capture_output=True, text=True, timeout=15)
        if _result.returncode == 0 and _result.stdout.strip():
            _os.environ["PROXY_API_KEY"] = _result.stdout.strip()
            print(f"build-cli token refreshed: …{_result.stdout.strip()[-4:]}")
        else:
            print(f"⚠️  build-cli auth token failed: {_result.stderr.strip() or 'no output'}")
            print("    Run 'build-cli login' in a terminal, then restart the kernel.")
    else:
        print("⚠️  build-cli not found — export PROXY_API_KEY=$(build-cli auth token)")


from aieng.forecasting.evaluation import EvalSpec, evaluate
from boc_rate_decisions.data import DIRECTION_SERIES_ID, build_boc_service
from boc_rate_decisions.press_releases import PressReleaseStore
from boc_rate_decisions.rationale_eval import evaluate_result_alignment


STATCAN_CACHE = ROOT / "data" / "statcan"
FRED_CACHE = ROOT / "data" / "fred"
SPECS_DIR = ROOT / "implementations" / "boc_rate_decisions" / "specs"
# Anchor the press-release cache to the repo root (notebook cwd is the use-case dir).
PRESS_RELEASE_CACHE = ROOT / "data" / "reports" / "boc_press_releases"

svc = build_boc_service(statcan_cache_dir=STATCAN_CACHE, fred_cache_dir=FRED_CACHE)
_as_of = datetime.now(tz=timezone.utc).replace(tzinfo=None)
direction_df = svc.get_series(DIRECTION_SERIES_ID, as_of=_as_of)

store = PressReleaseStore.from_cache(PRESS_RELEASE_CACHE)
print(f"Cached press releases: {len(store)}")
if len(store) == 0:
    print("No releases cached — run:  uv run python scripts/fetch_boc_press_releases.py")

build-cli token refreshed: …5EWw
Cached press releases: 13


---
## 2. Generate the traced runs to evaluate

Only methods that produce a `rationale` (the agent and the reasoning-enabled
LLMP) can be alignment-scored; the baselines are skipped automatically.

The judge reads each forecast **from its Langfuse trace**, so a traced run must
exist. `evaluate()` runs the predictors live over the protected post-2025 eval
window (`boc_rate_direction_eval.yaml`, 12 meetings Jan 2025 – Jun 2026) — the
same honest origins as notebook 02 §10 — emitting a fresh trace per origin, each
stamped with the structured forecast. There's no cache to go stale: every run
re-traces, so section 3 always has live traces to read.

> Running these two reasoning predictors over all 12 origins is ~24 model calls;
> it re-runs each time the cell executes. The accuracy scoreboard is computed and
> budgeted separately in notebook 02 §10 — this notebook only adds the *process*
> (alignment) verdict on top of the same traces.

In [2]:
from boc_rate_decisions.analyst_agent import build_boc_agent_predictor, build_boc_basic_config
from boc_rate_decisions.predictors import build_llmp_direction


# Model for BOTH reasoning predictors. Sonnet is the default; switch to haiku
# for faster, cheaper runs. The LLM-as-judge in §3 always uses the advanced
# model regardless of this choice.
MODEL = "anthropic/claude-sonnet-4-6[1m]"
# MODEL = "anthropic/claude-haiku-4-5-20251001"  # lite / cheaper

# Run the reasoning predictors over the PROTECTED POST-2025 eval window — the same
# honest origins as notebook 02 §10 (boc_rate_direction_eval.yaml: 12 meetings,
# Jan 2025 – Jun 2026, at/after the model's ~Jan 2025 cutoff). evaluate() runs each
# predictor live, emitting a fresh Langfuse trace per origin (each stamped with the
# structured forecast). Unlike cached_backtest there's no cache to go stale: every
# run re-traces, so the judge in section 3 always has live traces to read.
with (SPECS_DIR / "boc_rate_direction_eval.yaml").open() as f:
    spec = EvalSpec.model_validate(yaml.safe_load(f))

llmp = build_llmp_direction(model=MODEL, reasoning_effort=None)
agent = build_boc_agent_predictor(build_boc_basic_config(model=MODEL))
PREDICTOR_LABELS = {llmp.predictor_id: "LLMP direction", agent.predictor_id: "Agent (basic)"}

results = {}
for predictor in [llmp, agent]:
    # tracker=None: a side-channel eval runs unbudgeted and does not spend the
    # spec's max_runs accuracy-eval budget (mirrors notebook 02 §10).
    results[predictor.predictor_id] = evaluate(predictor=predictor, spec=spec, data_service=svc, tracker=None)
print(f"Loaded results ({MODEL}):", ", ".join(PREDICTOR_LABELS[p] for p in results))


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/pro

---
## 3. Judge each trace and push scores

For every trace the evaluator fetches it from Langfuse (polling briefly, since
ingestion is async), reads the stamped forecast, and runs one LLM-as-judge call
(advanced model). The judge scores *alignment only*; correctness comes from the
realised decision, and the two combine into `right_for_right_reasons`. With
`PUSH_TO_LANGFUSE = True` the verdict is written straight back to the trace as a
numeric `rationale_alignment` score and a categorical `right_for_right_reasons`
score, so it shows up alongside the trace in the Langfuse UI.

In [3]:
PUSH_TO_LANGFUSE = True  # write rationale_alignment + right_for_right_reasons scores back to each trace

frames = [
    evaluate_result_alignment(result, store, direction_df, push_to_langfuse=PUSH_TO_LANGFUSE)
    for result in results.values()
]
nonempty = [f for f in frames if not f.empty]
alignment = pd.concat(nonempty, ignore_index=True) if nonempty else pd.DataFrame()

if alignment.empty:
    print(
        "Scored 0 forecasts. Check that (1) Langfuse tracing is configured so the section 2 run emitted "
        "traces (LANGFUSE_* keys in .env), and (2) press releases are cached for these meetings "
        "(run scripts/fetch_boc_press_releases.py)."
    )
else:
    alignment["label"] = alignment["predictor_id"].map(PREDICTOR_LABELS)
    print(f"Scored {len(alignment)} rationale-bearing forecast(s).\n")
    summary = alignment.groupby("label").agg(
        n=("alignment_score", "size"),
        mean_alignment=("alignment_score", "mean"),
        correct_aligned=("right_for_right_reasons", lambda s: int((s == "correct_aligned").sum())),
    )
    print(summary.to_string())


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

Scored 12 rationale-bearing forecast(s).

                n  mean_alignment  correct_aligned
label                                             
Agent (basic)  12        0.615833                7


---
## 4. Per-meeting verdicts

Rendered as markdown (not crammed into a figure). Each verdict links to its
Langfuse trace when one is available.

In [4]:
if alignment.empty:
    print("Nothing to show — see the message above.")
else:
    for _, row in alignment.sort_values(["meeting_date", "label"]).iterrows():
        signals = ", ".join(row["key_signal_overlap"]) if row["key_signal_overlap"] else "—"
        trace = f"  ·  [trace]({row['langfuse_trace_url']})" if row.get("langfuse_trace_url") else ""
        display(
            Markdown(
                f"**{row['label']} — {row['meeting_date'].date()}**{trace}  \n"
                f"predicted **{row['predicted_label']}** · realised **{row['realized_label']}** · "
                f"alignment **{row['alignment_score']:.2f}** · _{row['right_for_right_reasons']}_\n\n"
                f"Signal overlap: {signals}\n\n"
                f"{row['justification']}\n\n---"
            )
        )

**Agent (basic) — 2025-01-29**  ·  [trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/52224dcfe2e6f48013f931001f12df38)  
predicted **cut** · realised **cut** · alignment **0.82** · _correct_aligned_

Signal overlap: Active easing cycle with consecutive cuts since June 2024, Labour market softening (unemployment rate elevated at 6.7%), CPI inflation close to 2% target, removing constraint on cutting, Shift toward more gradual/measured pace (25bps cut), Bond market yields signaling further accommodation

The forecaster correctly identified the core drivers the Bank cited: an active easing cycle totalling substantial cumulative cuts since June, labour market softness with elevated unemployment, and inflation near the 2% target allowing continued accommodation. The Bank's release confirms a 25bps cut with language about 'substantial cumulative reduction' and gradual strengthening ahead, matching the forecaster's expectation of a shift toward gradualism. The forecaster did not anticipate the significant new element in the Bank's release — the end of quantitative tightening and restart of asset purchases — nor the prominent tariff uncertainty framing, which were material parts of the Bank's communication but absent from the rationale.

---

**Agent (basic) — 2025-03-12**  ·  [trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/d2a0575aa046e09ab176677248da0259)  
predicted **cut** · realised **cut** · alignment **0.78** · _correct_aligned_

Signal overlap: Six consecutive cuts / sustained easing cycle momentum, CPI inflation close to 2% target (inflation not a barrier to cuts), Labour market softening / unemployment concerns, U.S. tariff threat uncertainty creating complex risk environment, Policy rate approaching neutral range (more deliberate approach), Bond yields easing / market pricing for further cuts, Governor Macklem signaling more gradual approach

The forecaster correctly identified the dominant drivers the Bank cited: inflation near target, labour market warning signs, and above all the pervasive uncertainty from U.S. tariff threats restraining spending and investment — all of which the Bank explicitly named as justification for the 25bp cut. The forecaster also correctly flagged the stagflationary complexity of tariffs (upward inflation pressure vs. downward growth pressure), which mirrors the Bank's own language about 'carefully assessing downward pressures on inflation from a weaker economy and upward pressures from higher costs.' The main gap is that the forecaster underweighted the tariff-driven growth slowdown as a decisive cut trigger, instead treating it more as a source of hold-risk, whereas the Bank treated the trade-conflict uncertainty as the primary reason to cut rather than pause.

---

**Agent (basic) — 2025-04-16**  ·  [trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/dee5b6cd62867285fd087cc7e4e96cf6)  
predicted **cut** · realised **hold** · alignment **0.72** · _incorrect_aligned_

Signal overlap: US tariff uncertainty flagged explicitly as a major downside risk to Canadian growth, Policy rate inside neutral range warranting more cautious pace, CPI inflation near 2% target, Labour market softening (employment declined, hiring plans slowing), Bank proceeding carefully given uncertainty, consistent with optionality argument

The forecaster correctly identified the key tension between the easing cycle momentum and the pause-inducing factors, with US tariff uncertainty and the rate being inside the neutral range being the most prominent shared signals. The Bank's decision to hold was driven primarily by the unprecedented uncertainty around US trade policy making it 'unusually challenging' to project outcomes, which the forecaster explicitly flagged as a reason for 'optionality rather than a committed cut.' However, the forecaster underweighted this uncertainty factor relative to the easing cycle momentum, assigning only 39% probability to a hold, whereas the Bank's language about 'pervasive uncertainty' and 'proceeding carefully' reflects a more decisive pause rationale than the forecaster anticipated. The inflation picture diverged slightly — the Bank noted inflation at 2.3% with upside risks from tariffs, while the forecaster characterized inflation as essentially at target with a dovish tilt.

---

**Agent (basic) — 2025-06-04**  ·  [trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/1718313e83e68d543cd09dc7d9e4cd8d)  
predicted **hold** · realised **hold** · alignment **0.82** · _correct_aligned_

Signal overlap: April 16, 2025 hold decision signaling assessment/pause phase, US-Canada tariff/trade uncertainty elevated, Labour market softening, unemployment rising, CPI inflation slightly above 2% target (excluding carbon tax effects), Policy rate within neutral range reducing urgency for cuts, BoC gradualism and proceeding carefully

The forecaster correctly identified the dominant drivers that the Bank cited: elevated US tariff uncertainty, a softening labour market (unemployment at 6.9%), and inflation slightly above target on an underlying basis (2.3% ex-taxes), all leading to a hold while gathering more information. The forecaster's emphasis on the April pause as a signal of deliberate assessment mode aligns directly with the Bank's stated rationale of 'proceeding carefully' and needing 'more information on US trade policy.' The one modest misalignment is that the forecaster cited CPI at ~2.32% as mildly constraining cuts, while the Bank's release highlighted headline CPI at 1.7% (carbon tax removal) but noted unexpected firmness in core/underlying measures — the directional signal is the same but the framing differs slightly.

---

**Agent (basic) — 2025-07-30**  ·  [trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/389ec0a031deeb54e96886d45f7da276)  
predicted **hold** · realised **hold** · alignment **0.72** · _correct_aligned_

Signal overlap: Two consecutive holds — Bank is in assessment/pause mode, US tariff and trade uncertainty persists as a downside risk to Canadian growth, CPI inflation near 2% target, Labour market softening / unemployment rate rising, Policy rate near neutral range, limiting urgency to cut further, Easing cycle context — hike essentially off the table

The forecaster correctly identified the dominant drivers of the hold decision: ongoing US tariff uncertainty, inflation near target, and a softening labour market, all of which feature prominently in the Bank's release. The Bank's explicit rationale — 'still high uncertainty, the Canadian economy showing some resilience, and ongoing pressures on underlying inflation' — aligns well with the forecaster's framing of institutional gradualism and data-dependence near neutral. However, the forecaster underweighted the Bank's concern about upward inflation pressures from tariff pass-through (underlying inflation at ~2.5%) and the GDP contraction narrative, and slightly overstated the dovish case by citing CPI at ~1.73% when the Bank reported June CPI at 1.9% with underlying inflation around 2.5%, making the inflation picture less unambiguously dovish than the forecaster suggested.

---

...

**Agent (basic) — 2025-12-10**  ·  [trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/5d8cbf02a046c503cf058dc119849a0e)  
predicted **hold** · realised **hold** · alignment **0.82** · _correct_aligned_

Signal overlap: Policy rate at neutral range proximity (~2.25%), raising threshold for further cuts, Inflation above 2% target (Bank cites core inflation 2.5%-3%, CPI at 2.2%), Labour market showing modest improvement, not emergency deterioration, Data dependence and assessment of cumulative prior cuts before further easing, Willingness to pause within easing cycle

The forecaster's primary drivers — neutral rate proximity, above-target inflation, and willingness to pause mid-cycle — closely mirror the Bank's stated reasoning that 'the current policy rate is at about the right level' given inflation near target and the need to assess cumulative impacts. The Bank's emphasis on core inflation remaining in the 2.5%-3% range aligns with the forecaster's concern about above-target inflation reducing urgency for cuts. The forecaster's labour market signal (modest softening, not emergency-level) also aligns with the Bank's characterisation of 'some signs of improvement' in employment, though the Bank noted a declining unemployment rate (6.5%) which is slightly more positive than the forecaster's framing of modest deterioration.

---

**Agent (basic) — 2026-01-28**  ·  [trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/7717d3658b3fd0f9381d8e92f55cf6cc)  
predicted **hold** · realised **hold** · alignment **0.62** · _correct_aligned_

Signal overlap: Hold decision after recent easing cycle pause, Inflation near 2% target (forecaster cited ~2.22%, Bank cited 2.4% with core easing toward 2.5%), Labour market improving (employment rising, though Bank notes elevated unemployment at 6.8%), Policy rate at neutral range as natural pause point, Gradualist/data-dependent approach supporting hold, Bond market not pricing imminent cuts

The forecaster correctly identified the hold outcome and cited several overlapping signals including inflation near target, the pause pattern after consecutive cuts, and institutional gradualism — all consistent with the Bank's statement that 'the current policy rate remains appropriate.' However, the forecaster's rationale diverged on key drivers: the Bank emphasized US trade policy uncertainty, tariff disruptions, stalled Q4 GDP growth, and geopolitical risks as central concerns, while the forecaster highlighted improving labour market momentum and a 2-year yield spread as primary signals — the Bank actually noted elevated unemployment at 6.8% and weak hiring intentions, contradicting the forecaster's 'improving labour market reduces downside risk' framing. The forecaster's reasoning was directionally correct but missed the dominant uncertainty narrative around US protectionism that the Bank placed front and center.

---

**Agent (basic) — 2026-03-18**  ·  [trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/11e30ba01d2afabd23a36b4d80e35faa)  
predicted **hold** · realised **hold** · alignment **0.38** · _correct_misaligned_

Signal overlap: Two consecutive holds signal deliberate pause in easing cycle, Policy rate near neutral range reducing urgency to cut further, Inflation running above/near 2% target

The forecaster correctly identified the hold outcome and cited some overlapping signals (consecutive holds, inflation near target, neutral rate proximity), but missed the dominant drivers in the Bank's actual release: a major geopolitical shock (Middle East war) causing energy price spikes, tightened financial conditions, and a weaker-than-expected Q4 GDP contraction. The forecaster's labour market signal was directly contradicted by the Bank, which noted employment gains were 'largely reversed' and unemployment rose to 6.7% — the opposite of the forecaster's 'tightening labour market' narrative. The Bank's decision was framed around balancing downside growth risks against upside inflation risks from the conflict, a framework entirely absent from the forecaster's rationale.

---

**Agent (basic) — 2026-04-29**  ·  [trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/1e717d6ef1bcec873bc8cc908edd14a7)  
predicted **hold** · realised **hold** · alignment **0.42** · _correct_misaligned_

Signal overlap: Three consecutive holds signaling a pause in the easing cycle, Policy rate near neutral range viewed as appropriate, Labour market conditions (soft but not deteriorating dramatically), Inflation near 2% target (though forecaster cited below-target while Bank cited above-target at 2.4%), Institutional gradualism and hold decision

The forecaster correctly identified the hold outcome and cited relevant signals like the extended pause, policy rate near neutral, and labour market conditions. However, the forecaster's inflation narrative was directionally wrong — they cited CPI ~1.78% (below target) as a mild hold-consistent signal, while the Bank's release showed CPI at 2.4% and rising toward ~3%, driven by sharply higher energy prices from the Iran war. The Bank's primary drivers were geopolitical uncertainty (Middle East conflict), energy price shocks, US tariff impacts, and upside inflation risks — none of which featured in the forecaster's rationale, representing a significant gap in the key drivers cited.

---

**Agent (basic) — 2026-06-10**  ·  [trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/9bde588cf75166b1535c647387bc52a2)  
predicted **hold** · realised **hold** · alignment **0.52** · _correct_aligned_

Signal overlap: Four consecutive holds establishing a pause phase, Policy rate near neutral range reducing urgency for further cuts, Inflation above 2% target (forecaster cited ~2.39%, Bank cited 2.8%), Global trade/tariff uncertainty weighing on Canadian growth outlook, Labour market stability (unemployment not sharply deteriorating), Hold as the baseline given data-dependent pause

The forecaster correctly identified the hold outcome and several relevant signals: the pause phase after the easing cycle, inflation above target, trade/tariff uncertainty, and a data-dependent stance. However, the forecaster's reasoning diverged significantly from the Bank's actual drivers — the Bank's release was dominated by the Middle East conflict, elevated oil prices pushing CPI to 2.8%, and supply chain disruptions, none of which featured in the forecaster's rationale. The forecaster cited a +0.72pp yield spread as an anti-cut signal, but the Bank noted 'loosened financial conditions' and 'buoyant equity markets,' pointing in a different direction. The forecaster's inflation estimate (~2.39%) also substantially undershot the Bank's reported 2.8%, meaning the inflation picture was more complex than anticipated.

---

---
## 5. Langfuse scores — review

The `rationale_alignment` and `right_for_right_reasons` scores were pushed to each
trace in section 3 (when `PUSH_TO_LANGFUSE = True`). This table summarises what
landed and links to each trace, so the verdicts are one click from the traces and
dashboards — a step toward closing the agent feedback loop. The **Result** column
(✅/❌) marks whether the predicted direction matched the actual decision —
*accuracy*, distinct from *alignment* (was the reasoning sound), so you can spot
the revealing cases: right for the wrong reasons (✅ + low alignment) and wrong
for sound reasons (❌ + high alignment).

> **Read the numbers with the window in mind.** This is **11 meetings per method**
> (Jan 2025 – Jun 2026) with **no hikes** — mostly holds and a few cuts. That's
> enough to *see* a model gap (switch `MODEL` in §2 to `anthropic/claude-haiku-4-5-20251001` for
> a cheaper comparison), but too
> small to *rank* models with confidence. Treat it as directional, not decisive.

In [5]:
if alignment.empty:
    print("Nothing scored — see section 3.")
else:
    n_total = len(alignment)
    n_pushed = int(alignment["langfuse_scored"].sum())
    correct_mask = alignment["predicted_label"] == alignment["realized_label"]
    n_correct = int(correct_mask.sum())

    table = [
        "| Method | Meeting | Result | Pred → Real | Alignment | Pushed | Trace |",
        "|---|---|:--:|---|---:|:--:|---|",
    ]
    for _, row in alignment.sort_values(["meeting_date", "label"]).iterrows():
        result = "✅" if row["predicted_label"] == row["realized_label"] else "❌"
        link = f"[open trace]({row['langfuse_trace_url']})" if row.get("langfuse_trace_url") else "—"
        pushed = "✅" if row.get("langfuse_scored") else "—"
        table.append(
            f"| {row['label']} | {row['meeting_date'].date()} | {result} | "
            f"{row['predicted_label']} → {row['realized_label']} | "
            f"{row['alignment_score']:.2f} | {pushed} | {link} |"
        )

    header = (
        f"**Langfuse — `rationale_alignment`**  \n"
        f"scored **{n_total}** · correct **{n_correct}/{n_total}** "
        f"(✅ = predicted direction matched the decision) · pushed **{n_pushed}**"
    )
    display(Markdown(header + "\n\n" + "\n".join(table)))

    if not PUSH_TO_LANGFUSE:
        display(
            Markdown(
                "_`PUSH_TO_LANGFUSE = False` in section 3 — set it `True` to write the scores. "
                "Trace links are clickable either way._"
            )
        )

**Langfuse — `rationale_alignment`**  
scored **12** · correct **9/12** (✅ = predicted direction matched the decision) · pushed **12**

| Method | Meeting | Result | Pred → Real | Alignment | Pushed | Trace |
|---|---|:--:|---|---:|:--:|---|
| Agent (basic) | 2025-01-29 | ✅ | cut → cut | 0.82 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/52224dcfe2e6f48013f931001f12df38) |
| Agent (basic) | 2025-03-12 | ✅ | cut → cut | 0.78 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/d2a0575aa046e09ab176677248da0259) |
| Agent (basic) | 2025-04-16 | ❌ | cut → hold | 0.72 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/dee5b6cd62867285fd087cc7e4e96cf6) |
| Agent (basic) | 2025-06-04 | ✅ | hold → hold | 0.82 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/1718313e83e68d543cd09dc7d9e4cd8d) |
| Agent (basic) | 2025-07-30 | ✅ | hold → hold | 0.72 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/389ec0a031deeb54e96886d45f7da276) |
| Agent (basic) | 2025-09-17 | ❌ | hold → cut | 0.25 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/7342ffdbbc8b254704db6b110b933c53) |
| Agent (basic) | 2025-10-29 | ❌ | hold → cut | 0.52 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/45b150d9712932e850ab3b997644d29c) |
| Agent (basic) | 2025-12-10 | ✅ | hold → hold | 0.82 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/5d8cbf02a046c503cf058dc119849a0e) |
| Agent (basic) | 2026-01-28 | ✅ | hold → hold | 0.62 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/7717d3658b3fd0f9381d8e92f55cf6cc) |
| Agent (basic) | 2026-03-18 | ✅ | hold → hold | 0.38 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/11e30ba01d2afabd23a36b4d80e35faa) |
| Agent (basic) | 2026-04-29 | ✅ | hold → hold | 0.42 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/1e717d6ef1bcec873bc8cc908edd14a7) |
| Agent (basic) | 2026-06-10 | ✅ | hold → hold | 0.52 | ✅ | [open trace](https://us.cloud.langfuse.com/project/cmqqnym0k0g4bad0cxlbpnnu3/traces/9bde588cf75166b1535c647387bc52a2) |